# Parsing Reddit posts
Emilio Lehoucq - 2/20/24

## Importing libraries

In [ ]:
import pandas as pd
import nltk
print(f'nltk version: {nltk.__version__}')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = nltk.corpus.stopwords.words('english')

nltk version: 3.8.1


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Defining custom function to parse text

In [ ]:
def text_parser(text):
  '''
  Function to parse text.

  Input: text (string)
  Output: tuple with list of unigrams, bigrams, and trigrams

  Dependencies: nltk (downloading punkt, stopwords [english], wordnet)
  '''
  tokens = nltk.word_tokenize(str(text)) # to avoid TypeError: expected string or bytes-like object
  unigrams = [token.lower() for token in tokens if token.isalnum() and token.lower() not in stop_words] # https://www.adamsmith.haus/python/answers/how-to-remove-all-punctuation-marks-with-nltk-in-python
  bigrams = list(nltk.bigrams(unigrams))
  trigrams = list(nltk.trigrams(unigrams))
  return unigrams, bigrams, trigrams

## Reading data

In [ ]:
df = pd.read_csv('reddit_mindfulness.csv', index_col = 0)

## Parsing posts and titles

In [ ]:
# Creating new columns for parsed text
df['title_unigrams'] = None
df['title_bigrams'] = None
df['title_trigrams'] = None
df['selftext_unigrams'] = None
df['selftext_bigrams'] = None
df['selftext_trigrams'] = None

# Indexes
IDX_UNIGRAMS = 0
IDX_BIGRAMS = 1
IDX_TRIGRAMS = 2

# Parsing text and populating new columns
for i, row in df.iterrows():
  parsed_title = text_parser(row['title'])
  parsed_selftext = text_parser(row['selftext'])
  df.at[i, 'title_unigrams'] = parsed_title[IDX_UNIGRAMS]
  df.at[i, 'title_bigrams'] = parsed_title[IDX_BIGRAMS]
  df.at[i, 'title_trigrams'] = parsed_title[IDX_TRIGRAMS]
  df.at[i, 'selftext_unigrams'] = parsed_selftext[IDX_UNIGRAMS]
  df.at[i, 'selftext_bigrams'] = parsed_selftext[IDX_BIGRAMS]
  df.at[i, 'selftext_trigrams'] = parsed_selftext[IDX_TRIGRAMS]

I decided not to lemmatize or stem. It could have been nice to do it. For example, it could have been nice to have "meditate," "meditation," and "meditating" to reduce to the same form. However, the out-of-the-box lemmatizer and stemmer in NLTK didn't seem to do a great job for some of the domain-specific words (run the code below to see).

In [ ]:
# tests_lemmatize_stem = [
#     "meditation",
#     "meditate",
#     "meditated",
#     "meditating",
#     "mindfulness",
#     "mindful",
#     "aware",
#     "awareness",
#     "loving-kindness",
#     "loving",
#     "love",
#     "compassion",
#     "compassionate",
#     "breathing",
#     "breathe",
#     "breath",
#     "attention",
#     "focusing",
#     "focus",
#     "attentive",
#     "gentle",
#     "kind"
#     ]

# for test in tests_lemmatize_stem:
#   print(f'Original word: {test}')
#   print(f'Lemma n: {nltk.WordNetLemmatizer().lemmatize(test, pos = "n")}')
#   print(f'Lemma v: {nltk.WordNetLemmatizer().lemmatize(test, pos = "v")}')
#   print(f'Lemma a: {nltk.WordNetLemmatizer().lemmatize(test, pos = "a")}')
#   print(f'Lemma r: {nltk.WordNetLemmatizer().lemmatize(test, pos = "r")}')
#   print(f'Lemma s: {nltk.WordNetLemmatizer().lemmatize(test, pos = "s")}')
#   print(f'Stem: {nltk.PorterStemmer().stem(test)}\n')

## Exporting data to CSV

Note: I only looked at a couple of examples. I have yet to explore the parsed text in detail.

In [ ]:
df.to_csv('reddit_mindfulness_parsed.csv')